# Fine-Tuning Whisper-small for Audio Dropout
## 1.Configuration
---

In [ ]:
!pip install audiomentations
!pip install jiwer
!pip install peft
!pip install --upgrade torchao


In [ ]:
from huggingface_hub import login

# Configuration

# torch.set_num_threads(1)
# torch.set_num_interop_threads(1)
dataset_samples = 1000
mask_ms = 300
model_folder = "whisper_dropout_300_2.5"
# for each audio clip total_mask determines total droupouts depending on audio length
# to make all of the audio dropouts proportional to each other
# total_corruptions = max(1, int(duration_sec / total_masks))
total_masks = 2.5

# os.environ["HF_TOKEN"] = ""
# login(token=os.environ["HF_TOKEN"])



## 2. Audio Processing
---

In [ ]:
import io
import os
import random
import soundfile as sf
from datasets import load_dataset, Dataset, DatasetDict, Audio
from audiomentations import TimeMask, Compose as AudioCompose

output_dir = "processed_audio"
os.makedirs(output_dir, exist_ok=True)

# define which id's go where before corrupting to keep split between
# clean -> clean training and fully corrupted test / validate
clean_ratio = 0.2
all_ids = list(range(dataset_samples))
random.shuffle(all_ids)

train_split = int(dataset_samples * 0.8)
val_split = int(dataset_samples * 0.9)

train_ids_set = set(all_ids[:train_split])
val_ids_set = set(all_ids[train_split:val_split])
test_ids_set = set(all_ids[val_split:])

train_records = []
val_records = []
test_records = []

records = []

stream = load_dataset(
    "librispeech_asr",
    "clean",
    split="train.100",
    streaming=True
)

stream = stream.cast_column("audio", Audio(decode=False))

corrupt_audio = AudioCompose([
    TimeMask(min_band_part=0.0, max_band_part=0.0, p=1.0)  # placeholder
])

for idx, sample in enumerate(stream):
    if idx >= dataset_samples:
        break

    # 1. Decode audio
    flac_bytes = sample["audio"]["bytes"]
    clean_audio, sr = sf.read(io.BytesIO(flac_bytes), dtype="float32")
    text = sample["text"]

    # 2. Save clean audio
    clean_path = os.path.join(output_dir, f"{idx}_clean.wav")
    sf.write(clean_path, clean_audio, sr)

    # 3. Corruption Logic
    is_test_or_val = idx in val_ids_set or idx in test_ids_set
    will_stay_clean = (not is_test_or_val) and (random.random() < clean_ratio)

    corrupted = None # Initialize to avoid NameError

    if will_stay_clean:
        corrupted_path = clean_path  # Re-use the clean file path
    else:
        corrupted = clean_audio.copy()
        mask_samples = int((mask_ms / 1000.0) * sr)
        total_samples = len(clean_audio)
        fraction = mask_samples / total_samples

        duration_sec = total_samples / sr
        total_corruptions = max(1, int(duration_sec / total_masks))

        for _ in range(total_corruptions):
            corrupt_audio.transforms[0].min_band_part = fraction
            corrupt_audio.transforms[0].max_band_part = fraction

            corrupted = corrupt_audio(samples=corrupted, sample_rate=sr)

        # 4. Save corrupted audio ONLY if we actually created it
        corrupted_path = os.path.join(output_dir, f"{idx}_corrupted.wav")
        sf.write(corrupted_path, corrupted, sr)

    # 5. Store metadata
    current_record = {
        "id": f"{idx}",
        "base_id": str(idx),
        "text": text,
        "clean_path": clean_path,
        "corrupted_path": corrupted_path,
        "sampling_rate": sr,
        "is_negative": will_stay_clean
    }

    # Append to the split lists correctly
    if idx in train_ids_set:
        train_records.append(current_record)
    elif idx in val_ids_set:
        val_records.append(current_record)
    elif idx in test_ids_set:
        test_records.append(current_record)

    # 6. Free RAM explicitly
    del clean_audio
    if corrupted is not None:
        del corrupted

# 5. Create the DatasetDict
dataset = DatasetDict({
    "train": Dataset.from_list(train_records),
    "validation": Dataset.from_list(val_records),
    "test": Dataset.from_list(test_records)
})

dataset


### 2.1 Save and create template for human evaluation

In [ ]:
import pandas as pd
import shutil

# Select random samples specifically from the test split
test_set = list(dataset["test"])

# Get unique base_ids:
human_eval_samples = random.sample(test_set, 10)

selected_base_ids = [str(r["base_id"]) for r in human_eval_samples]

print("Selected base IDs for human evaluation:")
for i, bid in enumerate(selected_base_ids):
    print(f"{i}: {bid}")

print(f"\nTotal human-eval records: {len(human_eval_samples)}")

# Save CSV template
template_df = pd.DataFrame({
    "id": selected_base_ids,
    "human_transcript": [""] * len(selected_base_ids)
})
template_df.to_csv("human_eval_template.csv", index=False)
print("\nTemplate saved as human_eval_template.csv")

# Save audio
audio_dir = "human_eval_audio"
if os.path.exists(audio_dir):
    shutil.rmtree(audio_dir)
os.makedirs(audio_dir, exist_ok=True)

for sample in human_eval_samples:
    sr = sample["sampling_rate"]

    # Save clean
    clean_audio, _ = sf.read(sample["clean_path"])
    sf.write(
        os.path.join("human_eval_audio", f"{sample['base_id']}_clean.wav"),
        clean_audio,
        sr
    )

    # Save corrupted
    corrupted_audio, _ = sf.read(sample["corrupted_path"])
    sf.write(
        os.path.join("human_eval_audio", f"{sample['base_id']}_corrupted.wav"),
        corrupted_audio,
        sr
    )

print("Saved human-eval audio files to human_eval_audio/")

## 3. LoRA fine tuning on Whisper Small model
---
### 3.1 Load model

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration, TrainingArguments, Trainer
import torch

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")


### 3.2 Preprocess data

In [ ]:
def preprocess(batch):
    audio, sr = sf.read(batch["corrupted_path"], dtype="float32")

    inputs = processor(audio, sampling_rate=sr)
    batch["input_features"] = inputs["input_features"][0]

    batch["labels"] = processor(text=batch["text"]).input_ids

    return batch


processed = dataset.map(
    preprocess,
    batched=False,
    remove_columns=[]
)

### 3.3 Define collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Extract and pad input_features
        input_features = [f["input_features"] for f in features]
        batch_inputs = self.processor.feature_extractor.pad(
            {"input_features": input_features},
            return_tensors="pt"
        )

        # Extract and pad labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        batch_labels = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = batch_labels["input_ids"].masked_fill(
            batch_labels["input_ids"] == self.processor.tokenizer.pad_token_id, -100
        )

        return {
            "input_features": batch_inputs["input_features"],
            "labels": labels,
        }

data_collator = WhisperDataCollator(processor=processor)

### 3.4 Compute Metrics

In [ ]:
from jiwer import wer
from transformers import TrainerCallback

def compute_metrics(eval_pred):
    pred_ids, labels = eval_pred

    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)

    # decode
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(labels, skip_special_tokens=True)

    # compute WER
    wer_score = wer(label_str, pred_str)

    return {
        "wer": wer_score
    }

class DualMetricEarlyStopping(TrainerCallback):
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.best_wer = float("inf")
        self.counter = 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        loss = metrics.get("eval_loss")
        wer = metrics.get("eval_wer") or metrics.get("wer")

        improved_loss = loss < (self.best_loss - self.min_delta)
        improved_wer = wer < (self.best_wer - self.min_delta)

        if improved_loss or improved_wer:
            if improved_loss:
                self.best_loss = loss
            if improved_wer:
                self.best_wer = wer
            self.counter = 0
        else:
            self.counter += 1

        if self.counter >= self.patience:
            control.should_training_stop = True

        return control



### 3.5 LoRA config and run trainer

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import numpy as np

# Load Base Model
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# PEFT setup
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=512,
    lora_alpha=1024,
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj"],
    lora_dropout=0,
    bias="none",
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

# Training args
training_args = Seq2SeqTrainingArguments(
    output_dir=model_folder,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-5,
    warmup_steps=50,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    logging_steps=10,
    eval_strategy="epoch",
    predict_with_generate=True,
    save_strategy="no",
    label_names=["labels"],
)

callbacks = [DualMetricEarlyStopping(patience=5, min_delta=0.0)]

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed["train"],
    eval_dataset=processed["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)

trainer.train()

for name, param in model.named_parameters():
    if "lora" in name:
        print(name, param.abs().mean().item())

model.save_pretrained("./" + model_folder)
processor.save_pretrained("./" + model_folder)

## 4. Results
---
### 4.1 Model inferences

In [ ]:
from jiwer import wer, cer
import re

from peft import PeftModel

# Config
device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "./" + model_folder

# "human" or "full_test"
mode = "human"

# Load models
processor = WhisperProcessor.from_pretrained(model_path)

# Pretrained
model_pre = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(device)
model_pre.config.forced_decoder_ids = None

# Finetuned (LoRA)
base = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(device)

model_ft = PeftModel.from_pretrained(base, model_path).to(device)
model_ft.config.forced_decoder_ids = None

def normalize(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

# Prep test samples
if mode == "full_test":
    test_samples = dataset["test"]
    test_samples = [dict(x) for x in test_samples]

# Optional human transcripts
human_lookup = None
if mode == "human":
    human_eval_df = pd.read_csv("human_eval_template.csv")
    human_lookup = dict(zip(
        human_eval_df["id"].astype(str).str.strip(),
        human_eval_df["human_transcript"]
    ))

def run_eval(samples, use_human_labels=False):
    results = []

    for sample in samples:
        sample_id = str(sample["id"])
        clean_text = normalize(sample["text"])

        # Load corrupted audio
        audio, sr = sf.read(sample["corrupted_path"], dtype="float32")

        # Convert to Whisper features
        input_features = processor(
            audio,
            sampling_rate=sr,
            return_tensors="pt"
        ).input_features.to(device)

        # Pretrained inference
        with torch.no_grad():
            pred_ids_pre = model_pre.generate(input_features)

        pred_pre = normalize(
            processor.batch_decode(pred_ids_pre, skip_special_tokens=True)[0]
        )

        # Finetuned inference
        with torch.no_grad():
            pred_ids_ft = model_ft.generate(input_features)

        pred_ft = normalize(
            processor.batch_decode(pred_ids_ft, skip_special_tokens=True)[0]
        )

        # Human transcript (optional)
        human_pred = None
        if use_human_labels:
            if sample_id not in human_lookup:
                print(f"No human transcript found for ID: {sample_id}")
                continue
            human_pred = normalize(human_lookup[sample_id])

        # Store results
        row = {
            "id": sample_id,
            "clean": clean_text,

            "pretrained_pred": pred_pre,
            "finetuned_pred": pred_ft,

            "wer_pretrained": wer(clean_text, pred_pre),
            "cer_pretrained": cer(clean_text, pred_pre),

            "wer_finetuned": wer(clean_text, pred_ft),
            "cer_finetuned": cer(clean_text, pred_ft),
        }

        if use_human_labels:
            row.update({
                "human_pred": human_pred,
                "wer_human": wer(clean_text, human_pred),
                "cer_human": cer(clean_text, human_pred),
            })

        results.append(row)

    return pd.DataFrame(results)

# Run mode
if mode == "human":
    results_df = run_eval(human_eval_samples, use_human_labels=True)

elif mode == "full_test":
    results_df = run_eval(test_samples, use_human_labels=False)

print(results_df.head())


### 4.2 Detailed summary

In [ ]:
print("\n--- PERFORMANCE SUMMARY ---")

possible_cols = [
    "wer_pretrained", "wer_finetuned", "wer_human",
    "cer_pretrained", "cer_finetuned", "cer_human"
]

summary_cols = [c for c in possible_cols if c in results_df.columns]

print(results_df[summary_cols].mean())


print("\n--- DETAILED RESULTS ---\n")

for i in range(len(results_df)):
    row = results_df.iloc[i]
    print(f"--- Example {i} ---")
    print(f"ID:              {row['id']}")
    print(f"Clean (Truth):   {row['clean']}")
    print(f"Pretrained Said: {row['pretrained_pred']}")
    print(f"Finetuned Said:  {row['finetuned_pred']}")

    if "human_pred" in results_df.columns:
        print(f"Human Said:      {row['human_pred']}")

    print(f"WER Pretrained:  {row['wer_pretrained']:.2f}")
    print(f"WER Finetuned:   {row['wer_finetuned']:.2f}")

    if "wer_human" in results_df.columns:
        print(f"WER Human:       {row['wer_human']:.2f}")

    print(f"CER Pretrained:  {row['cer_pretrained']:.2f}")
    print(f"CER Finetuned:   {row['cer_finetuned']:.2f}")

    if "cer_human" in results_df.columns:
        print(f"CER Human:       {row['cer_human']:.2f}")

    print()


### 4.3 Save results to a file

In [ ]:
possible_cols = [
    "wer_pretrained", "wer_finetuned", "wer_human",
    "cer_pretrained", "cer_finetuned", "cer_human"
]

summary_cols = [c for c in possible_cols if c in results_df.columns]

summary_text = []
summary_text.append("\n--- PERFORMANCE SUMMARY ---\n")
summary_text.append(str(results_df[summary_cols].mean()))
summary_text.append("\n")

with open("performance_summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_text))

print("Saved: performance_summary.txt")

lines = []
lines.append("\n--- DETAILED RESULTS ---\n")

for i in range(len(results_df)):
    row = results_df.iloc[i]

    lines.append(f"--- Example {i} ---")
    lines.append(f"ID:              {row['id']}")
    lines.append(f"Clean (Truth):   {row['clean']}")
    lines.append(f"Pretrained Said: {row['pretrained_pred']}")
    lines.append(f"Finetuned Said:  {row['finetuned_pred']}")

    if "human_pred" in results_df.columns:
        lines.append(f"Human Said:      {row['human_pred']}")

    lines.append(f"WER Pretrained:  {row['wer_pretrained']:.2f}")
    lines.append(f"WER Finetuned:   {row['wer_finetuned']:.2f}")

    if "wer_human" in results_df.columns:
        lines.append(f"WER Human:       {row['wer_human']:.2f}")

    lines.append(f"CER Pretrained:  {row['cer_pretrained']:.2f}")
    lines.append(f"CER Finetuned:   {row['cer_finetuned']:.2f}")

    if "cer_human" in results_df.columns:
        lines.append(f"CER Human:       {row['cer_human']:.2f}")

    lines.append("")  # blank line between examples

with open("detailed_results.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("Saved: detailed_results.txt")
